# Практикум 7. Решающие деревья

Сопровождает интерактивную лабораторию №7 («Решающие деревья»). Тема курса: структура дерева, критерии ветвления и правила

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Дерево как набор правил

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

Xtr_t, Xte_t, ytr, yte = train_test_split(df.text, df.label, test_size=0.3, stratify=df.label, random_state=0)
vec = TfidfVectorizer(max_features=300)
Xtr = vec.fit_transform(Xtr_t); Xte = vec.transform(Xte_t)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(export_text(tree, feature_names=list(vec.get_feature_names_out())))
plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=vec.get_feature_names_out(), class_names=tree.classes_, filled=True, fontsize=8)
plt.show()

## 2. Критерии ветвления: Gini и энтропия

In [ ]:
for crit in ["gini", "entropy"]:
    t = DecisionTreeClassifier(criterion=crit, max_depth=6, random_state=0).fit(Xtr, ytr)
    print(crit, "train:", round(t.score(Xtr, ytr), 3), "test:", round(t.score(Xte, yte), 3))

## 3. Переобучение: глубина дерева

In [ ]:
depths = range(1, 21)
tr, te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    tr.append(t.score(Xtr, ytr)); te.append(t.score(Xte, yte))
plt.plot(depths, tr, label="обучение"); plt.plot(depths, te, label="тест")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Глубина и переобучение"); plt.show()

## 4. Важность слов

In [ ]:
t = DecisionTreeClassifier(max_depth=6, random_state=0).fit(Xtr, ytr)
imp = pd.Series(t.feature_importances_, index=vec.get_feature_names_out()).sort_values(ascending=False).head(12)
imp[::-1].plot.barh(); plt.title("Важность слов в дереве"); plt.show()

## Задания

1. Подберите `min_samples_leaf` и `max_depth`, при которых точность на тесте максимальна.
2. Попробуйте `ccp_alpha` (отсечение ветвей, cost-complexity pruning). Как меняется размер дерева?
3. Запишите словами правило для листа с самой высокой чистотой.